In [47]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from agents.sql_agent import *
from agents.orchestrator import *

In [49]:
max_sql_retries = 3
chat_history = 5
api_key_gemini = os.getenv("GEMINI_KEY")
api_key_gpt = os.getenv("MY_API_KEY")

In [50]:
db_name = "finance.db"

db_description = """
Tables:

1. outcome
Columns:
- Date (DATE): expense date in YYYY-MM-DD format. There may be multiple expenses on the same day
- Value (FLOAT): amount of money spent (in polish zloty)
- Category (TEXT): type of expense. Options: czynsz, trip, services, yummy, restraunt, food, present, clothes, transport, outdoors, holiday, work, marta, projeb, health, electronics, house, cosmetics, education, charity, unknown, remont, jewellery
- Was planned (TEXT): whether the outcome was planned ('yes', 'no')
- Description (TEXT): a subcategory of category

2. income
Columns:
- Date (DATE): same as above
- Value (FLOAT): amount of money received
- Category (TEXT): type of income. Options: same as for outcome + salary, stypendium
- Constant (TEXT): whether the income is constant ('yes', 'no')
- Description (TEXT): same as above
"""

In [61]:
model = ChatOpenAI(model="gpt-3.5-turbo", api_key=api_key_gpt)
# model = ChatGoogleGenerativeAI(model="gemini-flash-latest", google_api_key=api_key_gemini)
sql_agent = SQLAgent(model, db_name, db_description, max_sql_retries)
buddy = DataBuddyAgent(model, sql_agent, chat_history)

In [62]:
user_input = "I want all expenses from zabka from 2025"
user_input = "was i spending more than i year in 2025?"
user_input = "ile miesiecnie srednio trace na jedzenie"
user_input = "сколько денег уходит в среднем месяц на косметику c 2026 года"
# user_input = "What's the weather in London now?"
# user_input = "i want to see how many i spend in this year so far on clothes"

In [63]:
buddy = DataBuddyAgent(model, sql_agent, chat_history)
buddy.chat("сколько денег уходит в среднем месяц на косметику c 2026 года")

,avg_monthly_spending_on_cosmetics
0,59.236667
1,215.040000
2,30.317500


{'intent': 'CREATE_NEW_QUERY',
 'answer': "The result is quite large, so I've displayed the data instead. You can ask me to summarize it or narrow down your request.",
 'last query': "SELECT AVG(Value) AS avg_monthly_spending_on_cosmetics\nFROM outcome\nWHERE Category = 'cosmetics'\nAND Date >= '2026-01-01'\nGROUP BY strftime('%Y-%m', Date)",
 'token spent': 1150}

In [64]:
buddy.chat("не, сколько в месяц я трачу на косметику с 2026 года?")

{'intent': 'CREATE_NEW_QUERY',
 'answer': 'In 2026, you spent a total of $1159.14 on cosmetics for the entire year.',
 'last query': "SELECT SUM(Value) AS total_spent_on_cosmetics\nFROM outcome\nWHERE Category LIKE 'cosmetics'\nAND Date >= '2026-01-01'\nAND Date < '2027-01-01';",
 'token spent': 2063}

In [65]:
buddy.chat("не, я хочу суммарно для каждого месяца, а потом с этого среднюю")

,Month,Total
0,01,99.744382
1,01,44386.250000
2,02,98.411382
3,02,69084.790000
4,03,97.160372
5,03,66652.015000
6,04,62.726122
7,04,36067.520000
8,05,119.620899
9,05,63877.560000


{'intent': 'CREATE_NEW_QUERY',
 'answer': "The result is quite large, so I've displayed the data instead. You can ask me to summarize it or narrow down your request.",
 'last query': "SELECT \n    strftime('%m', Date) AS Month,\n    SUM(Value) AS Total\nFROM outcome\nGROUP BY Month\nUNION\nSELECT \n    strftime('%m', Date) AS Month,\n    AVG(Value) AS Average\nFROM outcome\nGROUP BY Month\nORDER BY Month;",
 'token spent': 3317}

In [66]:
buddy.chat("епта, че за хуйня, просто посчитай сколько в месяц я трачу на косметику")

{'intent': 'CREATE_NEW_QUERY',
 'answer': 'The table shows that the total amount spent on cosmetics in the month of October 2022 is not available or zero.\nWithout the specific data for October 2022, we are unable to provide the exact amount spent on cosmetics for that month.',
 'last query': "SELECT SUM(Value) AS total_cosmetics_spent_per_month\nFROM outcome\nWHERE Category LIKE 'cosmetics'\nAND Date BETWEEN '2022-10-01' AND '2022-10-31';",
 'token spent': 4338}

In [67]:
buddy.chat("епта, че за хуйня, просто посчитай сколько в месяц я трачу на косметику в 2026")

{'intent': 'CREATE_NEW_QUERY',
 'answer': "In 2026, you spent a total of $1159.14 on cosmetics for the year. This information was obtained by summing up all expenses categorized as 'cosmetics' between January 1, 2026, and December 31, 2026.",
 'last query': "SELECT \n    SUM(Value) AS total_cosmetics_expense\nFROM \n    outcome\nWHERE \n    Category LIKE 'cosmetics' \n    AND Date BETWEEN '2026-01-01' AND '2026-12-31';",
 'token spent': 5429}

In [ ]:
a

In [ ]:
while True:
    user_input = input("Your question (or 'quit'): ")
    if user_input.lower() == "quit":
        break
    res = buddy.chat(user_input)
    print("\nAnswer:\n", res["answer"])
    print("Query:\n", res["last query"])
    print("Intent:\n", res["intent"])
    print("Tokens spent:\n", res["token spent"])

In [ ]:
def query_executor(query, db="finance.db"):
    """Executes SQL query and returns a DataFrame. Returns error if invalid."""
    try:
        with sqlite3.connect(db) as conn:
            df = pd.read_sql_query(query, conn)
        return df, None
    except Exception as e:
        return None, str(e)

In [ ]:
query_executor("SELECT * FROM outcome WHERE Category = 'clothes' AND Date LIKE '2025-01%'")